In [19]:
import pandas as pd
from Bio import SeqIO
import re
import os
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import traceback

# --- CẤU HÌNH ---
FASTA_PATH = r"D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa"

WINDOW = 500
TARGET_LEN = 1001 # 15 + 1 (đột biến) + 15
PAD_CHAR = 'X'

# --- HÀM TIỆN ÍCH ---

def load_ensembl_protein_fasta(path):
    """
    Kế thừa logic load genome: Map từ Protein ID (ENSP) sang Protein Sequence.
    """
    mapping = {}
    print(f"🧬 Đang nạp FASTA từ {path}...")
    for record in SeqIO.parse(path, "fasta"):
        # Trích xuất ENST từ header: ... transcript:ENST00000641515.2 ...
        match = re.search(r'transcript:(ENST\d+)', record.description)
        if match:
            enst_id = match.group(1)
            mapping[enst_id] = str(record.seq).upper()
    print(f"✅ Đã nạp {len(mapping)} mã transcript.")
    return mapping

def normalize_centered_protein(seq, center_idx, target_len, pad_char='X'):
    """
    Kế thừa hoàn toàn logic 'Symmetric Crop/Pad' từ notebook DNA của bạn.
    """
    half = target_len // 2
    start = center_idx - half
    end = center_idx + half + 1
    
    pad_left = max(0, -start)
    pad_right = max(0, end - len(seq))
    
    crop_left = max(0, start)
    crop_right = min(len(seq), end)
    
    final_seq = (pad_char * pad_left) + seq[crop_left:crop_right] + (pad_char * pad_right)
    return final_seq[:target_len] # Đảm bảo luôn đủ 31

In [20]:
# ==============================================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN & THAM SỐ
# ==============================================================================
PARQUET_PATH = r"D:\variant_data\train1_full_seq_final.parquet"
# Ghi ra file tạm để tránh xung đột đọc/ghi trên cùng 1 file
TEMP_OUTPUT_PATH = r"D:\variant_data\train1_full_seq_final_temp.parquet"
chunk_size = 100000

protein_dict = load_ensembl_protein_fasta(FASTA_PATH)
print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")

# ==============================================================================
# 2. HÀM TIỆN ÍCH XỬ LÝ 1 BIẾN THỂ (Tối ưu tốc độ, loại bỏ iterrows)
# ==============================================================================
def process_single_variant(ensp_val, enst_val, prot_pos_val, aa_val):
    try:
        ensp_id = str(ensp_val).split(".")[0] if pd.notna(ensp_val) else None
        enst_id = str(enst_val).split(".")[0] if pd.notna(enst_val) else None

        full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
        if not full_ref_protein:
            return None, None

        pos_str = str(prot_pos_val).split("-")[0]
        if pos_str in ("-", "nan", ""):
            return None, None
        pos_0based = int(pos_str) - 1
        if pos_0based < 0 or pos_0based >= len(full_ref_protein):
            return None, None

        aa_change = str(aa_val).strip()
        if "/" in aa_change:
            ref_aa_part, alt_aa_part = aa_change.split("/", 1)
        elif aa_change not in ("-", "nan", ""):
            ref_aa_part = alt_aa_part = aa_change
        else:
            return None, None

        actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
        if actual_ref_aa != ref_aa_part:
            return None, None

        alt_full_protein = (
            full_ref_protein[:pos_0based]
            + alt_aa_part
            + full_ref_protein[pos_0based + len(ref_aa_part) :]
        )

        if "*" in alt_full_protein or "X" in alt_aa_part:
            stop_idx = alt_full_protein.find("*")
            if stop_idx == -1:
                stop_idx = alt_full_protein.find("X", pos_0based)
            pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
            alt_full_protein = alt_full_protein[: stop_idx + 1] + (PAD_CHAR * pad_len)

        ref_seq = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
        alt_seq = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
        return ref_seq, alt_seq

    except Exception:
        return None, None

# ==============================================================================
# 3. VÒNG LẶP CHUNK-BY-CHUNK AN TOÀN VỚI PYARROW
# ==============================================================================
if os.path.exists(TEMP_OUTPUT_PATH):
    os.remove(TEMP_OUTPUT_PATH)

parquet_writer = None
parquet_file = pq.ParquetFile(PARQUET_PATH)

ref_col = f"prot_ref_seq_{TARGET_LEN}"
alt_col = f"prot_alt_seq_{TARGET_LEN}"

try:
    for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
        chunk = batch.to_pandas()

        # Dùng zip trích xuất thay cho iterrows() -> nhanh hơn ~20 lần
        results = [
            process_single_variant(ensp, enst, pos, aa)
            for ensp, enst, pos, aa in zip(
                chunk["ENSP"], chunk["Feature"], chunk["Protein_position"], chunk["Amino_acids"]
            )
        ]

        ref_seqs, alt_seqs = zip(*results)
        chunk[ref_col] = ref_seqs
        chunk[alt_col] = alt_seqs

        valid_chunk = chunk.dropna(subset=[ref_col]).copy()

        if len(valid_chunk) > 0:
            table = pa.Table.from_pandas(valid_chunk, preserve_index=False)

            if parquet_writer is None:
                parquet_writer = pq.ParquetWriter(TEMP_OUTPUT_PATH, table.schema)
            else:
                table = table.cast(parquet_writer.schema)

            parquet_writer.write_table(table)

        print(f" > Đã xử lý xong chunk {chunk_idx + 1} ({len(valid_chunk):,} dòng hợp lệ)...")

finally:
    if parquet_writer is not None:
        parquet_writer.close()
        print(f"[!] Đã đóng file Parquet an toàn.")

# Thay thế file cũ bằng file tạm sau khi đã ghi thành công toàn bộ
del parquet_file
os.replace(TEMP_OUTPUT_PATH, PARQUET_PATH)
print(f"🎉 Hoàn thành ghi đè file an toàn: {PARQUET_PATH}")

# ==============================================================================
# 4. KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# ==============================================================================
df = pd.read_parquet(PARQUET_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df[ref_col].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [{TARGET_LEN}])")

mid_idx = TARGET_LEN // 2
check_diff = df[df[ref_col].str[mid_idx] == df[alt_col].str[mid_idx]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm vị trí {mid_idx} giống hệt nhau: {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\train1_full_seq_final.parquet
 > Đã xử lý xong chunk 1 (100,000 dòng hợp lệ)...
 > Đã xử lý xong chunk 2 (4,385 dòng hợp lệ)...
[!] Đã đóng file Parquet an toàn.
🎉 Hoàn thành ghi đè file an toàn: D:\variant_data\train1_full_seq_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 104,385

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [1001] (Kỳ vọng: [1001])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm vị trí 500 giống hệt nhau: 0


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,prot_ref_seq_31,prot_alt_seq_31,prot_ref_seq_101,prot_alt_seq_101,prot_ref_seq_201,prot_alt_seq_201,prot_ref_seq_501,prot_alt_seq_501,prot_ref_seq_1001,prot_alt_seq_1001
0,1_1014042_G_A,446939.0,single nucleotide variant,NM_005101.4(ISG15):c.62G>A (p.Ser21Asn),9636.0,ISG15,HGNC:4053,Benign/Likely benign,0.0,"MONDO:MONDO:0014502,MedGen:C4015293,OMIM:61612...",...,TVKMLAGNEFQVSLSSSMSVSELKAQITQKI,TVKMLAGNEFQVSLSNSMSVSELKAQITQKI,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMGWDLTVKMLAGNEFQ...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMGWDLTVKMLAGNEFQ...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
1,1_1014228_G_A,389314.0,single nucleotide variant,NM_005101.4(ISG15):c.248G>A (p.Ser83Asn),9636.0,ISG15,HGNC:4053,Benign,0.0,"MedGen:CN169374|MONDO:MONDO:0014502,MedGen:C40...",...,GSTVLLVVDKCDEPLSILVRNNKGRSSTYEV,GSTVLLVVDKCDEPLNILVRNNKGRSSTYEV,TQKIGVHAFQQRLAVHPSGVALQDRVPLASQGLGPGSTVLLVVDKC...,TQKIGVHAFQQRLAVHPSGVALQDRVPLASQGLGPGSTVLLVVDKC...,XXXXXXXXXXXXXXXXXXMGWDLTVKMLAGNEFQVSLSSSMSVSEL...,XXXXXXXXXXXXXXXXXXMGWDLTVKMLAGNEFQVSLSSSMSVSEL...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
2,1_1014401_G_A,NaN,None,None,NaN,None,None,None,NaN,None,...,FEGKPLEDQLPLGEYGLKPLSTVFMNLRLRG,FEGKPLEDQLPLGEYSLKPLSTVFMNLRLRG,GRSSTYEVRLTQTVAHLKQQVSGLEGVQDDLFWLTFEGKPLEDQLP...,GRSSTYEVRLTQTVAHLKQQVSGLEGVQDDLFWLTFEGKPLEDQLP...,FQQRLAVHPSGVALQDRVPLASQGLGPGSTVLLVVDKCDEPLSILV...,FQQRLAVHPSGVALQDRVPLASQGLGPGSTVLLVVDKCDEPLSILV...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
3,1_1014471_G_C,446981.0,single nucleotide variant,NM_005101.4(ISG15):c.491G>C (p.Arg164Pro),9636.0,ISG15,HGNC:4053,Likely benign,0.0,"MONDO:MONDO:0014502,MedGen:C4015293,OMIM:61612...",...,FMNLRLRGGGTEPGGRSXXXXXXXXXXXXXX,FMNLRLRGGGTEPGGPSXXXXXXXXXXXXXX,LEGVQDDLFWLTFEGKPLEDQLPLGEYGLKPLSTVFMNLRLRGGGT...,LEGVQDDLFWLTFEGKPLEDQLPLGEYGLKPLSTVFMNLRLRGGGT...,GLGPGSTVLLVVDKCDEPLSILVRNNKGRSSTYEVRLTQTVAHLKQ...,GLGPGSTVLLVVDKCDEPLSILVRNNKGRSSTYEVRLTQTVAHLKQ...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
4,1_1020183_G_C,364282.0,single nucleotide variant,NM_198576.4(AGRN):c.11G>C (p.Arg4Pro),375790.0,AGRN,HGNC:329,Benign,0.0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",...,XXXXXXXXXXXXMAGRSHPGPLRPLLPLLVV,XXXXXXXXXXXXMAGPSHPGPLRPLLPLLVV,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104380,X_155511709_C_T,2821775.0,single nucleotide variant,NM_018196.4(TMLHE):c.722G>A (p.Arg241Gln),55217.0,TMLHE,HGNC:18308,Likely benign,0.0,MedGen:C3661900,...,DFSRGDTAYTKLALDRHTDTTYFQEPCGIQV,DFSRGDTAYTKLALDQHTDTTYFQEPCGIQV,FVENVPPTQEHTEKLAERISLIRETIYGRMWYFTSDFSRGDTAYTK...,FVENVPPTQEHTEKLAERISLIRETIYGRMWYFTSDFSRGDTAYTK...,QKQKVIQPRILWNAEIYQQAQVPSVDCQSFLETNEGLKKFLQNFLL...,QKQKVIQPRILWNAEIYQQAQVPSVDCQSFLETNEGLKKFLQNFLL...,XXXXXXXXXXMWYHRLSHLHSRLQDLLKGGVIYPALPQPNFKSLLP...,XXXXXXXXXXMWYHRLSHLHSRLQDL

In [21]:
PARQUET_PATH = r"D:\variant_data\train2_full_seq_final.parquet"
OUTPUT_PATH = r"D:\variant_data\train2_full_seq_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk[f'prot_ref_seq_{TARGET_LEN}'], chunk[f'prot_alt_seq_{TARGET_LEN}'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=[f'prot_ref_seq_{TARGET_LEN}']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
        else:
            table = table.cast(parquet_writer.schema)    
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df[f'prot_ref_seq_{TARGET_LEN}'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [{TARGET_LEN}])") 

check_diff = df[df[f'prot_ref_seq_{TARGET_LEN}'].str[TARGET_LEN//2] == df[f'prot_alt_seq_{TARGET_LEN}'].str[TARGET_LEN//2]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm {TARGET_LEN//2} giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\train2_full_seq_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\train2_full_seq_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 67,990

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [1001] (Kỳ vọng: [1001])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 500 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,prot_ref_seq_31,prot_alt_seq_31,prot_ref_seq_101,prot_alt_seq_101,prot_ref_seq_201,prot_alt_seq_201,prot_ref_seq_501,prot_alt_seq_501,prot_ref_seq_1001,prot_alt_seq_1001
0,1_1020183_G_C,364282.0,single nucleotide variant,NM_198576.4(AGRN):c.11G>C (p.Arg4Pro),375790.0,AGRN,HGNC:329,Benign,0.0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",...,XXXXXXXXXXXXMAGRSHPGPLRPLLPLLVV,XXXXXXXXXXXXMAGPSHPGPLRPLLPLLVV,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
1,1_1022383_C_G,NaN,None,None,NaN,None,None,None,NaN,None,...,RIFFVNPAPPYLWPAHKNELMLNSSLMRITL,RIFFVNPAPPYLWPAQKNELMLNSSLMRITL,DLVARESLLDGGNKVVISGFGDPLICDNQVSTGDTRIFFVNPAPPY...,DLVARESLLDGGNKVVISGFGDPLICDNQVSTGDTRIFFVNPAPPY...,GGTCPERALERREEEANVVLTGTVEEILNVDPVQHTYSCKVRVWRY...,GGTCPERALERREEEANVVLTGTVEEILNVDPVQHTYSCKVRVWRY...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
2,1_1035307_C_T,446942.0,single nucleotide variant,NM_198576.4(AGRN):c.494C>T (p.Pro165Leu),375790.0,AGRN,HGNC:329,Likely benign,0.0,"MONDO:MONDO:0014052,MedGen:C3808739,OMIM:61512...",...,EFCVEDKPGTHFTPVPPTPPDACRGMLCGFG,EFCVEDKPGTHFTPVLPTPPDACRGMLCGFG,FFVNPAPPYLWPAHKNELMLNSSLMRITLRNLEEVEFCVEDKPGTH...,FFVNPAPPYLWPAHKNELMLNSSLMRITLRNLEEVEFCVEDKPGTH...,SCKVRVWRYLKGKDLVARESLLDGGNKVVISGFGDPLICDNQVSTG...,SCKVRVWRYLKGKDLVARESLLDGGNKVVISGFGDPLICDNQVSTG...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
3,1_1041218_C_T,249307.0,single nucleotide variant,NM_198576.4(AGRN):c.773C>T (p.Thr258Ile),375790.0,AGRN,HGNC:329,Benign/Likely benign,0.0,"MedGen:CN169374|MONDO:MONDO:0014052,MedGen:C38...",...,GSRDPCSNVTCSFGSTCARSADGLTASCLCP,GSRDPCSNVTCSFGSICARSADGLTASCLCP,PVCGSDASTYSNECELQRAQCSQQRRIRLLSRGPCGSRDPCSNVTC...,PVCGSDASTYSNECELQRAQCSQQRRIRLLSRGPCGSRDPCSNVTC...,GTHFTPVPPTPPDACRGMLCGFGAVCEPNAEGPGRASCVCKKSPCP...,GTHFTPVPPTPPDACRGMLCGFGAVCEPNAEGPGRASCVCKKSPCP...,GPLRPLLPLLVVAACVLPGAGGTCPERALERREEEANVVLTGTVEE...,GPLRPLLPLLVVAACVLPGAGGTCPERALERREEEANVVLTGTVEE...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
4,1_1041583_A_G,133740.0,single nucleotide variant,NM_198576.4(AGRN):c.1058A>G (p.Gln353Arg),375790.0,AGRN,HGNC:329,Benign,0.0,MedGen:CN169374|MedGen:C3661900|MONDO:MONDO:00...,...,RRPEMLLRPESCPARQAPVCGDDGVTYENDC,RRPEMLLRPESCPARRAPVCGDDGVTYENDC,CARQENVFKKFDGPCDPCQGALPDPSRSCRVNPRTRRPEMLLRPES...,CARQENVFKKFDGPCDPCQGALPDPSRSCRVNPRTRRPEMLLRPES...,CSFGSTCARSADGLTASCLCPATCRGAPEGTVCGSDGADYPGECQL...,CSFGSTCARSADGLTASCLCPATCRGAPEGTVCGSDGADYPGECQL...,CDNQVSTGDTRIFFVNPAPPYLWPAHKNELMLNSSLMRITLRNLEE...,CDNQVSTGDTRIFFVNPAPPYLWPAHKNELMLNSSLMRITLRNLEE...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67985,X_155506983_C_T,4308127.0,single nucleotide variant,NM_018196.4(TMLHE):c.910G>A (p.Asp304Asn),55217.0,TMLHE,HGNC:18308,Likely benign,0.0,MedGen:CN169374,...,FELLSKVPLKHEYIEDVGECHNHMIGIGPVL,FELLSKVPLKHEYIENVGECHNHMIGIGPVL,IQVFHCLKHEGTGGRTLLVDGFYAAEQVLQKAPEEFELLSKVPLKH...,IQVFHCLKHEGTGGRTLLVDGFYAAEQVLQKAPEEFELLSKVPLKH...,KLAERISLIRETIYGRMWYFTSDFSRGDTAYTKLALDRHTDTTYFQ...,KLAERISLIRETIYGRMWYFTSDFSRGDTAYTKLALDRHTDTTYFQ...,QQHEDHFELKYANTVMRFDYVWLRDHCRSASCYNSKTHQRSLDTAS...,QQHEDHFELKYANTVMRFDYVWLRDHCR

In [22]:
PARQUET_PATH = r"D:\variant_data\train3_full_seq_final.parquet"
OUTPUT_PATH = r"D:\variant_data\train3_full_seq_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk[f'prot_ref_seq_{TARGET_LEN}'], chunk[f'prot_alt_seq_{TARGET_LEN}'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=[f'prot_ref_seq_{TARGET_LEN}']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
        else:
            table = table.cast(parquet_writer.schema)    
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df[f'prot_ref_seq_{TARGET_LEN}'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [{TARGET_LEN}])") 

check_diff = df[df[f'prot_ref_seq_{TARGET_LEN}'].str[TARGET_LEN//2] == df[f'prot_alt_seq_{TARGET_LEN}'].str[TARGET_LEN//2]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm {TARGET_LEN//2} giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\train3_full_seq_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\train3_full_seq_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 23,208

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [1001] (Kỳ vọng: [1001])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 500 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,prot_ref_seq_31,prot_alt_seq_31,prot_ref_seq_101,prot_alt_seq_101,prot_ref_seq_201,prot_alt_seq_201,prot_ref_seq_501,prot_alt_seq_501,prot_ref_seq_1001,prot_alt_seq_1001
0,12_53315378_C_T,2048536.0,single nucleotide variant,NM_015665.6(AAAS):c.356G>A (p.Arg119Gln),8086.0,AAAS,HGNC:13666,Likely benign,0.0,MedGen:C3661900,...,FEWVKTASGWALALCRWASSLHGSLFPHLSL,FEWVKTASGWALALCQWASSLHGSLFPHLSL,FIHHREQVWKRCINIWRDVGLFGVLNEIANSEEEVFEWVKTASGWA...,FIHHREQVWKRCINIWRDVGLFGVLNEIANSEEEVFEWVKTASGWA...,YEHNNELVTGSSYESPPPDFRGQWINLPVLQLTKDPLKTPGRLDHG...,YEHNNELVTGSSYESPPPDFRGQWINLPVLQLTKDPLKTPGRLDHG...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
1,12_53321403_G_C,325881.0,single nucleotide variant,NM_015665.6(AAAS):c.63C>G (p.His21Gln),8086.0,AAAS,HGNC:13666,Likely benign,0.0,MedGen:C3661900|,...,LFPPPPPRGQVTLYEHNNELVTGSSYESPPP,LFPPPPPRGQVTLYEQNNELVTGSSYESPPP,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMCSLGLFPPPPPRGQV...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMCSLGLFPPPPPRGQV...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
2,12_53314832_C_T,859992.0,single nucleotide variant,NM_015665.6(AAAS):c.464G>A (p.Arg155His),8086.0,AAAS,HGNC:13666,Pathogenic/Likely pathogenic,1.0,"MedGen:C3661900|MONDO:MONDO:0009279,MedGen:C02...",...,IAEFAQVTNWSSCCLRVFAWHPHTNKFAVAL,IAEFAQVTNWSSCCLHVFAWHPHTNKFAVAL,EWVKTASGWALALCRWASSLHGSLFPHLSLRSEDLIAEFAQVTNWS...,EWVKTASGWALALCRWASSLHGSLFPHLSLRSEDLIAEFAQVTNWS...,LKTPGRLDHGTRTAFIHHREQVWKRCINIWRDVGLFGVLNEIANSE...,LKTPGRLDHGTRTAFIHHREQVWKRCINIWRDVGLFGVLNEIANSE...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
3,12_53321423_G_T,20083.0,single nucleotide variant,NM_015665.6(AAAS):c.43C>A (p.Gln15Lys),8086.0,AAAS,HGNC:13666,Pathogenic,1.0,"MONDO:MONDO:0009279,MedGen:C0271742,OMIM:23155...",...,XMCSLGLFPPPPPRGQVTLYEHNNELVTGSS,XMCSLGLFPPPPPRGKVTLYEHNNELVTGSS,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMCSLGLFPPP...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMCSLGLFPPP...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
4,16_70258214_C_T,466634.0,single nucleotide variant,NM_001605.3(AARS1):c.1996G>A (p.Val666Ile),16.0,AARS1,HGNC:20,Likely benign,0.0,"MONDO:MONDO:0018993,MedGen:C0270914,Orphanet:6...",...,KAEEIANEMIEAAKAVYTQDCPLAAAKAIQG,KAEEIANEMIEAAKAIYTQDCPLAAAKAIQG,RSVLGEADQKGSLVAPDRLRFDFTAKGAMSTQQIKKAEEIANEMIE...,RSVLGEADQKGSLVAPDRLRFDFTAKGAMSTQQIKKAEEIANEMIE...,AQVRGGYVLHIGTIYGDLKVGDQVWLFIDEPRRRPIMSNHTATHIL...,AQVRGGYVLHIGTIYGDLKVGDQVWLFIDEPRRRPIMSNHTATHIL...,DTYGFPVDLTGLIAEEKGLVVDMDGFEEERKLAQLKSQGKGAGGED...,DTYGFPVDLTGLIAEEKGLVVDMDGFEEERKLAQLKSQGKGAGGED...,ILPGNMKDNFWEMGDTGPCGPCSEIHYDRIGGRDAAHLVNQDDPNV...,ILPGNMKDNFWEMGDTGPCGPCSEIHYDRIGGRDAAHLVNQDDPNV...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23203,7_76425104_A_G,3409690.0,single nucleotide variant,NM_001110354.2(ZP3):c.140A>G (p.Gln47Arg),7784.0,ZP3,HGNC:13189,Likely benign,0.0,MedGen:C3661900|MedGen:CN169374,...,ASHPETSVQPVLVECQEATLMVMVSKDLFGT,ASHPETSVQPVLVECREATLMVMVSKDLFGT,XXXXMELSYRLFICLLLWGSTELCYPQPLWLLQGGASHPETSVQPV...,XXXXMELSYRLFICLLLWGSTELCYPQPLWLLQGGASHPETSVQPV...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

In [23]:
PARQUET_PATH = r"D:\variant_data\val_full_seq_final.parquet"
OUTPUT_PATH = r"D:\variant_data\val_full_seq_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk[f'prot_ref_seq_{TARGET_LEN}'], chunk[f'prot_alt_seq_{TARGET_LEN}'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=[f'prot_ref_seq_{TARGET_LEN}']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
        else:
            table = table.cast(parquet_writer.schema)    
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df[f'prot_ref_seq_{TARGET_LEN}'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [{TARGET_LEN}])") 

check_diff = df[df[f'prot_ref_seq_{TARGET_LEN}'].str[TARGET_LEN//2] == df[f'prot_alt_seq_{TARGET_LEN}'].str[TARGET_LEN//2]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm {TARGET_LEN//2} giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\val_full_seq_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\val_full_seq_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 13,701

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [1001] (Kỳ vọng: [1001])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 500 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,prot_ref_seq_31,prot_alt_seq_31,prot_ref_seq_101,prot_alt_seq_101,prot_ref_seq_201,prot_alt_seq_201,prot_ref_seq_501,prot_alt_seq_501,prot_ref_seq_1001,prot_alt_seq_1001
0,4_1022218_C_T,720682.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.95C>T (p.Ala32Val),53834.0,FGFRL1,HGNC:3693,Benign,0.0,MedGen:C3661900|,...,GAFPPAAAARGPPKMADKVVPRQVARLGRTV,GAFPPAAAARGPPKMVDKVVPRQVARLGRTV,XXXXXXXXXXXXXXXXXXXMTPSPLLLLLLPPLLLGAFPPAAAARG...,XXXXXXXXXXXXXXXXXXXMTPSPLLLLLLPPLLLGAFPPAAAARG...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
1,4_1022236_G_A,1949144.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.113G>A (p.Arg38Gln),53834.0,FGFRL1,HGNC:3693,Likely benign,0.0,MedGen:C3661900,...,AAARGPPKMADKVVPRQVARLGRTVRLQCPV,AAARGPPKMADKVVPQQVARLGRTVRLQCPV,XXXXXXXXXXXXXMTPSPLLLLLLPPLLLGAFPPAAAARGPPKMAD...,XXXXXXXXXXXXXMTPSPLLLLLLPPLLLGAFPPAAAARGPPKMAD...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
2,4_1023924_G_A,734353.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.541G>A (p.Asp181Asn),53834.0,FGFRL1,HGNC:3693,Benign/Likely benign,0.0,"MedGen:C3661900|MONDO:MONDO:0008684,MedGen:C19...",...,SSVRLKCVASGHPRPDITWMKDDQALTRPEA,SSVRLKCVASGHPRPNITWMKDDQALTRPEA,SSSGGQEDPASQQWARPRFTQPSKMRRRVIARPVGSSVRLKCVASG...,SSSGGQEDPASQQWARPRFTQPSKMRRRVIARPVGSSVRLKCVASG...,PQGLKVKQVEREDAGVYVCKATNGFGSLSVNYTLVVLDDISPGKES...,PQGLKVKQVEREDAGVYVCKATNGFGSLSVNYTLVVLDDISPGKES...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
3,4_1024917_C_A,1154604.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.1085C>A (p.Pro362Gln),53834.0,FGFRL1,HGNC:3693,Benign,0.0,MedGen:C3661900,...,SFRSAFLTVLPDPKPPGPPVASSSSATSLPW,SFRSAFLTVLPDPKPQGPPVASSSSATSLPW,WSRPDGSYLNKLLITRARQDDAGMYICLGANTMGYSFRSAFLTVLP...,WSRPDGSYLNKLLITRARQDDAGMYICLGANTMGYSFRSAFLTVLP...,GTTSFQCKVRSDVKPVIQWLKRVEYGAEGRHNSTIDVGGQKFVVLP...,GTTSFQCKVRSDVKPVIQWLKRVEYGAEGRHNSTIDVGGQKFVVLP...,YTLVVLDDISPGKESLGPDSSSGGQEDPASQQWARPRFTQPSKMRR...,YTLVVLDDISPGKESLGPDSSSGGQEDPASQQWARPRFTQPSKMRR...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
4,4_1024946_G_A,2724602.0,single nucleotide variant,NM_001004356.3(FGFRL1):c.1114G>A (p.Ala372Thr),53834.0,FGFRL1,HGNC:3693,Likely benign,0.0,MedGen:CN169374,...,PDPKPPGPPVASSSSATSLPWPVVIGIPAGA,PDPKPPGPPVASSSSTTSLPWPVVIGIPAGA,KLLITRARQDDAGMYICLGANTMGYSFRSAFLTVLPDPKPPGPPVA...,KLLITRARQDDAGMYICLGANTMGYSFRSAFLTVLPDPKPPGPPVA...,SDVKPVIQWLKRVEYGAEGRHNSTIDVGGQKFVVLPTGDVWSRPDG...,SDVKPVIQWLKRVEYGAEGRHNSTIDVGGQKFVVLPTGDVWSRPDG...,PGKESLGPDSSSGGQEDPASQQWARPRFTQPSKMRRRVIARPVGSS...,PGKESLGPDSSSGGQEDPASQQWARPRFTQPSKMRRRVIARPVGSS...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13696,22_50744150_G_A,3640163.0,single nucleotide variant,NM_001097.3(ACR):c.655G>A (p.Val219Ile),49.0,ACR,HGNC:126,Likely benign,0.0,MedGen:CN169374,...,IDLDLCNSTQWYNGRVQPTNVCAGYPVGKID,IDLDLCNSTQWYNGRIQPTNVCAGYPVGKID,GLPRGSQSCWVAGWGYIEEKAPRPSSILMEARVDLIDLDLCNSTQW...,GLPRGSQSCWVAGWGYIEEKAPRPSSILMEARVDLIDLDLCNSTQW...,PLQERYVEKIIIHEKYNSATEGNDIALVEITPPISCGRFIGPGCLP...,PLQERYVEKIIIHEKYNSATEGNDIALVEITPPISCGRFIGPGCLP...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMVEMLPTAILLVLA...,XXXXXXXXXXXX

In [24]:
PARQUET_PATH = r"D:\variant_data\test_full_seq_after_vep_final.parquet"
OUTPUT_PATH = r"D:\variant_data\test_full_seq_after_vep_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk[f'prot_ref_seq_{TARGET_LEN}'], chunk[f'prot_alt_seq_{TARGET_LEN}'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=[f'prot_ref_seq_{TARGET_LEN}']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
        else:
            table = table.cast(parquet_writer.schema)    
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df[f'prot_ref_seq_{TARGET_LEN}'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [{TARGET_LEN}])") 

check_diff = df[df[f'prot_ref_seq_{TARGET_LEN}'].str[TARGET_LEN//2] == df[f'prot_alt_seq_{TARGET_LEN}'].str[TARGET_LEN//2]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm {TARGET_LEN//2} giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\test_full_seq_after_vep_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\test_full_seq_after_vep_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 6,095

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [1001] (Kỳ vọng: [1001])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 500 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,PhenotypeIDS,...,prot_ref_seq_31,prot_alt_seq_31,prot_ref_seq_101,prot_alt_seq_101,prot_ref_seq_201,prot_alt_seq_201,prot_ref_seq_501,prot_alt_seq_501,prot_ref_seq_1001,prot_alt_seq_1001
0,10_180088_C_T,424656.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.76C>T (p.Arg26Trp),10771.0,ZMYND11,HGNC:16966,Pathogenic/Likely pathogenic,1.0,"MONDO:MONDO:0014486,MedGen:C4015167,OMIM:61608...",...,DTKAIQHLWAAIEIIRNQKQIANIDRITKYM,DTKAIQHLWAAIEIIWNQKQIANIDRITKYM,XXXXXXXXXXXXXXXXXXXXXXXXXMARLTKRRQADTKAIQHLWAA...,XXXXXXXXXXXXXXXXXXXXXXXXXMARLTKRRQADTKAIQHLWAA...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
1,10_237638_G_T,2038412.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.570G>T (p.Arg190Ser),10771.0,ZMYND11,HGNC:16966,Likely benign,0.0,"MedGen:C3661900|MeSH:D030342,MedGen:C0950123",...,DLNKKGKDNKHPMYRRLVHSAVDVPTIQEKV,DLNKKGKDNKHPMYRSLVHSAVDVPTIQEKV,WQCPVCRSIKKKNTNKQEMGTYLRFIVSRMKERAIDLNKKGKDNKH...,WQCPVCRSIKKKNTNKQEMGTYLRFIVSRMKERAIDLNKKGKDNKH...,DEIDWETENHDWYCFECHLPGEVLICDLCFRVYHSKCLSDEFRLRD...,DEIDWETENHDWYCFECHLPGEVLICDLCFRVYHSKCLSDEFRLRD...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
2,10_240913_C_G,2077177.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.774C>G (p.Cys258Trp),10771.0,ZMYND11,HGNC:16966,Pathogenic,1.0,MedGen:C3661900,...,MLYKDTCHELDELQLCKNCFYLSNARPDNWF,MLYKDTCHELDELQLWKNCFYLSNARPDNWF,GKYRSYEEFKADAQLLLHNTVIFYGADSEQADIARMLYKDTCHELD...,GKYRSYEEFKADAQLLLHNTVIFYGADSEQADIARMLYKDTCHELD...,MGTYLRFIVSRMKERAIDLNKKGKDNKHPMYRRLVHSAVDVPTIQE...,MGTYLRFIVSRMKERAIDLNKKGKDNKHPMYRRLVHSAVDVPTIQE...,RQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHGMHPKETT...,RQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHGMHPKETT...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
3,10_242031_A_C,NaN,None,None,NaN,None,None,None,NaN,None,...,NARPDNWFCYPCIPNHELVWAKMKGFGFWPA,NARPDNWFCYPCIPNPELVWAKMKGFGFWPA,YGADSEQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYP...,YGADSEQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYP...,KDNKHPMYRRLVHSAVDVPTIQEKVNEGKYRSYEEFKADAQLLLHN...,KDNKHPMYRRLVHSAVDVPTIQEKVNEGKYRSYEEFKADAQLLLHN...,IANIDRITKYMSRVHGMHPKETTRQLSLAVKDGLIVETLTVGCKGS...,IANIDRITKYMSRVHGMHPKETTRQLSLAVKDGLIVETLTVGCKGS...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
4,10_246823_C_G,3002871.0,single nucleotide variant,NM_001370100.5(ZMYND11):c.1008C>G (p.His336Gln),10771.0,ZMYND11,HGNC:16966,Benign,0.0,MedGen:C3661900,...,PSENIQDITVNIHRLHVKRSMGWKKACDELE,PSENIQDITVNIHRLQVKRSMGWKKACDELE,AKMKGFGFWPAKVMQKEDNQVDVRFFGHHHQRAWIPSENIQDITVN...,AKMKGFGFWPAKVMQKEDNQVDVRFFGHHHQRAWIPSENIQDITVN...,EQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYPCIPNH...,EQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYPCIPNH...,WLPGDEIDWETENHDWYCFECHLPGEVLICDLCFRVYHSKCLSDEF...,WLPGDEIDWETENHDWYCFECHLPGEVLICDLCFRVYHSKCLSDEF...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6090,20_64208061_G_A,728821.0,single nucleotide variant,NM_004535.3(MYT1):c.865G>A (p.Glu289Lys),4661.0,MYT1,HGNC:7622,Likely benign,0.0,MedGen:C3661900|,...,DEEEEEEEEEEEEEEEEEEEEEEEEEEEEEE,DEEEEEEEEEEEEEEKEEEEEEEEEEEEEEE,SLEDAASEESSKQKGILSHEEEDEEEEEEEEEEEEDEEEEEEEEEE...,SLEDAASEESSKQKGILSHEEEDEEEEEEEEEEEEDEEEEEEEEEE...,KPGPGIVHLLQEAAEGAASEEGEKGLFIQPEDAEEVVEVTTERSQD...,KPGPGIVHLLQEAAEGAASEEGEKGLFIQPEDAEEVVEVTTERSQD...,GHVRGKYSRHRSLQSCPLAKKRKLEGAEAEHLVSKRKSHPLKLALD...,GHVRGKYSRHRSLQSCPLAKKRKLEGAEAEHLVSKRKSHPLKLALD...,XX

In [25]:
PARQUET_PATH = r"D:\variant_data\clinvarhq_full_seq_after_vep_final.parquet"
OUTPUT_PATH = r"D:\variant_data\clinvarhq_full_seq_after_vep_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk[f'prot_ref_seq_{TARGET_LEN}'], chunk[f'prot_alt_seq_{TARGET_LEN}'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=[f'prot_ref_seq_{TARGET_LEN}']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
        else:
            table = table.cast(parquet_writer.schema)    
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df[f'prot_ref_seq_{TARGET_LEN}'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [{TARGET_LEN}])") 

check_diff = df[df[f'prot_ref_seq_{TARGET_LEN}'].str[TARGET_LEN//2] == df[f'prot_alt_seq_{TARGET_LEN}'].str[TARGET_LEN//2]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm {TARGET_LEN//2} giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\clinvarhq_full_seq_after_vep_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\clinvarhq_full_seq_after_vep_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 703

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [1001] (Kỳ vọng: [1001])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 500 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,CHROM,POS,REF,ALT,Label,ID,GeneInfo,CLNSIG,CLNREVSTAT,...,prot_ref_seq_31,prot_alt_seq_31,prot_ref_seq_101,prot_alt_seq_101,prot_ref_seq_201,prot_alt_seq_201,prot_ref_seq_501,prot_alt_seq_501,prot_ref_seq_1001,prot_alt_seq_1001
0,10_237638_G_T,chr10,237638,G,T,0,1980541,ZMYND11,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,DLNKKGKDNKHPMYRRLVHSAVDVPTIQEKV,DLNKKGKDNKHPMYRSLVHSAVDVPTIQEKV,WQCPVCRSIKKKNTNKQEMGTYLRFIVSRMKERAIDLNKKGKDNKH...,WQCPVCRSIKKKNTNKQEMGTYLRFIVSRMKERAIDLNKKGKDNKH...,DEIDWETENHDWYCFECHLPGEVLICDLCFRVYHSKCLSDEFRLRD...,DEIDWETENHDWYCFECHLPGEVLICDLCFRVYHSKCLSDEFRLRD...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
1,10_357852_G_C,chr10,357852,G,C,0,2758294,DIP2C,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,SGKRIAQASGRDLGQIEDNDQARKFLFLSEV,SGKRIAQASGRDLGQMEDNDQARKFLFLSEV,CNVLMCPHTCVTNLPKPRQKQPEIGPASVMVGNLVSGKRIAQASGR...,CNVLMCPHTCVTNLPKPRQKQPEIGPASVMVGNLVSGKRIAQASGR...,SRVLQAIDSIHQVGVYCLALVPANTLPKTPLGGIHLSETKQLFLEG...,SRVLQAIDSIHQVGVYCLALVPANTLPKTPLGGIHLSETKQLFLEG...,GAIMCSVKPDGVPQLCRTDEIGELCVCAVATGTSYYGLSGMTKNTF...,GAIMCSVKPDGVPQLCRTDEIGELCVCAVATGTSYYGLSGMTKNTF...,FKGWPKLLWFVTESKHLSKPPRDWFPHIKDANNDTAYIEYKTCKDG...,FKGWPKLLWFVTESKHLSKPPRDWFPHIKDANNDTAYIEYKTCKDG...
2,10_1086292_C_T,chr10,1086292,C,T,0,2279214,WDR37,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,AKTQPVVLGTASADHTALLWSIETGKCLVKY,AKTQPVVLGTASADHMALLWSIETGKCLVKY,TSKIVSSFKTTTSRAACQLVKEYIGHRDGIWDVSVAKTQPVVLGTA...,TSKIVSSFKTTTSRAACQLVKEYIGHRDGIWDVSVAKTQPVVLGTA...,RREIDTLNERLAAEGQAIDGAELSKGQLKTKASHSTSQLSQKLKTT...,RREIDTLNERLAAEGQAIDGAELSKGQLKTKASHSTSQLSQKLKTT...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
3,10_5102114_C_T,chr10,5102114,C,T,0,716665,AKR1C3,Likely_benign,"criteria_provided,_multiple_submitters,_no_con...",...,PGLKYKPVCNQVECHPYFNRSKLLDFCKSKD,PGLKYKPVCNQVECHLYFNRSKLLDFCKSKD,CTTWEAMEKCKDAGLAKSIGVSNFNRRQLEMILNKPGLKYKPVCNQ...,CTTWEAMEKCKDAGLAKSIGVSNFNRRQLEMILNKPGLKYKPVCNQ...,VRPALENSLKKAQLDYVDLYLIHSPMSLKPGEELSPTDENGKVIFD...,VRPALENSLKKAQLDYVDLYLIHSPMSLKPGEELSPTDENGKVIFD...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
4,10_8064041_G_A,chr10,8064041,G,A,1,3384342,GATA3,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,GRECVNCGATSTPLWRRDGTGHYLCNACGLY,GRECVNCGATSTPLWQRDGTGHYLCNACGLY,VPEYSSGLFPPSSLLGGSPTGFGCKSRPKARSSTEGRECVNCGATS...,VPEYSSGLFPPSSLLGGSPTGFGCKSRPKARSSTEGRECVNCGATS...,ARQDEKECLKYQVPLPDSMKLESSHSRGSMTALGGASSSTHHPITT...,ARQDEKECLKYQVPLPDSMKLESSHSRGSMTALGGASSSTHHPITT...,THHPGLSHSYMDAAQYPLPEEVDVLFNIDGQGNHVPPYYGNSVRAT...,THHPGLSHSYMDAAQYPLPEEVDVLFNIDGQGNHVPPYYGNSVRAT...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
698,20_63488381_C_A,chr20,63488381,C,A,1,383531,EEF1A2,Likely_pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,GRFAVRDMRQTVAVGVIKNVEKKSGGAGKVT,GRFAVRDMRQTVAVGFIKNVEKKSGGAGKVT,LEDNPKSLKSGDAAIVEMVPGKPMCVESFSQYPPLGRFAVRDMRQT...,LEDNPKSLKSGDAAIVEMVPGKPMCVESFSQYPPLGRFAVRDMRQT...,AAQFTSQVIILNHPGQISAGYSPVIDCHTAHIACKFAELKEKIDRR...,AAQFTSQVIILNHPGQISAGYSPVIDCHTAHIACKFAELKEKIDRR...,TVPFVPISGWHGDNMLEPSPNMPWFKGWKVERKEGNASGVSLLEAL...,TVPFVPISGWHGDNMLEPSPNMPWFKGWKVERKEGNASGVSLLEAL...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
699,20_63495909_C_T,chr20,63495909,C,T,1,279803,EEF1A2,Pathogenic,"criteria_provided,_multiple_submitters,_no_con...",...,SLWKFETTKYYITIIDAPGHRDFIKNMITGT,SLWKF

In [26]:
PARQUET_PATH = r"D:\variant_data\proteingym_full_seq_after_vep_final.parquet"
OUTPUT_PATH = r"D:\variant_data\proteingym_full_seq_after_vep_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk[f'prot_ref_seq_{TARGET_LEN}'], chunk[f'prot_alt_seq_{TARGET_LEN}'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=[f'prot_ref_seq_{TARGET_LEN}']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
        else:
            table = table.cast(parquet_writer.schema)    
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df[f'prot_ref_seq_{TARGET_LEN}'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [{TARGET_LEN}])") 

check_diff = df[df[f'prot_ref_seq_{TARGET_LEN}'].str[TARGET_LEN//2] == df[f'prot_alt_seq_{TARGET_LEN}'].str[TARGET_LEN//2]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm {TARGET_LEN//2} giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\proteingym_full_seq_after_vep_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\proteingym_full_seq_after_vep_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 1,472

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [1001] (Kỳ vọng: [1001])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 500 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,CHROM,POS,REF,ALT,Label,protein,protein_sequence,mutant,mutated_sequence,...,prot_ref_seq_31,prot_alt_seq_31,prot_ref_seq_101,prot_alt_seq_101,prot_ref_seq_201,prot_alt_seq_201,prot_ref_seq_501,prot_alt_seq_501,prot_ref_seq_1001,prot_alt_seq_1001
0,10_180088_C_T,chr10,180088,C,T,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,R26W,MARLTKRRQADTKAIQHLWAAIEIIWNQKQIANIDRITKYMSRVHG...,...,DTKAIQHLWAAIEIIRNQKQIANIDRITKYM,DTKAIQHLWAAIEIIWNQKQIANIDRITKYM,XXXXXXXXXXXXXXXXXXXXXXXXXMARLTKRRQADTKAIQHLWAA...,XXXXXXXXXXXXXXXXXXXXXXXXXMARLTKRRQADTKAIQHLWAA...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
1,10_240913_C_G,chr10,240913,C,G,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,C258W,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,MLYKDTCHELDELQLCKNCFYLSNARPDNWF,MLYKDTCHELDELQLWKNCFYLSNARPDNWF,GKYRSYEEFKADAQLLLHNTVIFYGADSEQADIARMLYKDTCHELD...,GKYRSYEEFKADAQLLLHNTVIFYGADSEQADIARMLYKDTCHELD...,MGTYLRFIVSRMKERAIDLNKKGKDNKHPMYRRLVHSAVDVPTIQE...,MGTYLRFIVSRMKERAIDLNKKGKDNKHPMYRRLVHSAVDVPTIQE...,RQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHGMHPKETT...,RQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHGMHPKETT...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
2,10_242031_A_C,chr10,242031,A,C,1,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,H281P,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,NARPDNWFCYPCIPNHELVWAKMKGFGFWPA,NARPDNWFCYPCIPNPELVWAKMKGFGFWPA,YGADSEQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYP...,YGADSEQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYP...,KDNKHPMYRRLVHSAVDVPTIQEKVNEGKYRSYEEFKADAQLLLHN...,KDNKHPMYRRLVHSAVDVPTIQEKVNEGKYRSYEEFKADAQLLLHN...,IANIDRITKYMSRVHGMHPKETTRQLSLAVKDGLIVETLTVGCKGS...,IANIDRITKYMSRVHGMHPKETTRQLSLAVKDGLIVETLTVGCKGS...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
3,10_248502_G_A,chr10,248502,G,A,0,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,S465N,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,MLHRSTQTTNDGVCQSMCHDKYTKIFNDFKD,MLHRSTQTTNDGVCQNMCHDKYTKIFNDFKD,TEAVSSSQEIPTMPQPIEKVSVSTQTKKLSASSPRMLHRSTQTTND...,TEAVSSSQEIPTMPQPIEKVSVSTQTKKLSASSPRMLHRSTQTTND...,SKNEDRGEEEAESSISSTSNEQLKVTQEPRAKKGRRNQSVEPKKEE...,SKNEDRGEEEAESSISSTSNEQLKVTQEPRAKKGRRNQSVEPKKEE...,EFKADAQLLLHNTVIFYGADSEQADIARMLYKDTCHELDELQLCKN...,EFKADAQLLLHNTVIFYGADSEQADIARMLYKDTCHELDELQLCKN...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMARLTKRRQA...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMARLTKRRQA...
4,10_248559_C_T,chr10,248559,C,T,0,NP_006615.2,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,S484L,MARLTKRRQADTKAIQHLWAAIEIIRNQKQIANIDRITKYMSRVHG...,...,DKYTKIFNDFKDRMKSDHKRETERVVREALE,DKYTKIFNDFKDRMKLDHKRETERVVREALE,VSVSTQTKKLSASSPRMLHRSTQTTNDGVCQSMCHDKYTKIFNDFK...,VSVSTQTKKLSASSPRMLHRSTQTTNDGVCQSMCHDKYTKIFNDFK...,NEQLKVTQEPRAKKGRRNQSVEPKKEEPEPETEAVSSSQEIPTMPQ...,NEQLKVTQEPRAKKGRRNQSVEPKKEEPEPETEAVSSSQEIPTMPQ...,DSEQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYPCIP...,DSEQADIARMLYKDTCHELDELQLCKNCFYLSNARPDNWFCYPCIP...,XXXXXXXXXXXXXXXXXMARLTKRRQADTKAIQHLWAAIEIIRNQK...,XXXXXXXXXXXXXXXXXMARLTKRRQADTKAIQHLWAAIEIIRNQK...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1467,20_64207869_G_A,chr20,64207869,G,A,0,NP_004526.1,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,V225I,MSLENEDKRARTRSKALRGPPETTAADLSCPTPGCTGSGHVRGKYS...,...,GEKGLFIQPEDAEEVVEVTTERSQDLCPQSL,GEKGLFIQPEDAEEVIEVTTERSQDLCPQSL,AEETLVEEDLGQAAKPGPGIVHLLQEAAEGAASEEGEKGLFIQPED...,AEETLVEEDLGQAAKPGPGIVHLLQEAAEGAASEEGEKGLFIQPED...,IHRPETAEGRSPVKSHFGSNPIGSATASSKGSYSSYQGIIATSLLN...,IHRPETAEGRSPVKSHFGSNPIGSATASSKGSYSSYQGIIATSLLN...,XXXXXXXXXXXXXXXXXXXXXXXXXXMSLENEDKRARTRSKALRGP...,XXXXXXXXXXXXXXXXXXXXXXXXXXMSLENEDKRARTRSKALRGP...,XXXXXXXXXXXXXXXXXXXXX

In [27]:
PARQUET_PATH = r"D:\variant_data\uniprot_full_seq_after_vep_final.parquet"
OUTPUT_PATH = r"D:\variant_data\uniprot_full_seq_after_vep_final.parquet"

# (Giả định bạn đã có hàm load_ensembl_protein_fasta và normalize_centered_protein)
# LƯU Ý: Hàm load_ensembl_protein_fasta giờ nên dùng ENSP (ID đầu tiên trong header FASTA) làm key.
protein_dict = load_ensembl_protein_fasta(FASTA_PATH)

print(f"📊 Đang xử lý file Parquet: {PARQUET_PATH}")
chunk_size = 100000

# Khởi tạo ParquetWriter
parquet_writer = None

parquet_file = pq.ParquetFile(PARQUET_PATH)

for chunk_idx, batch in enumerate(parquet_file.iter_batches(batch_size=chunk_size)):
    # Chuyển batch của Arrow thành Pandas DataFrame
    chunk = batch.to_pandas()
    
    results = []
    
    for idx, row in chunk.iterrows():
        try:
            # 1. DÙNG ENSP ĐỂ MAP VỚI PROTEIN FASTA
            ensp_id = str(row['ENSP']).split('.')[0] if pd.notna(row['ENSP']) else None
            enst_id = str(row['Feature']).split('.')[0]
            
            # Thử tìm bằng ENSP trước, không có thì thử ENST
            full_ref_protein = protein_dict.get(ensp_id) or protein_dict.get(enst_id)
            
            if not full_ref_protein:
                results.append((None, None))
                continue
            
            # 2. Xử lý vị trí
            pos_str = str(row['Protein_position']).split('-')[0]
            if pos_str == '-' or pos_str == 'nan':
                results.append((None, None))
                continue
                
            pos_1based = int(pos_str)
            pos_0based = pos_1based - 1
            
            # 3. Xử lý Amino Acid (Ref/Alt)
            aa_change = str(row['Amino_acids']).strip()
            
            if '/' in aa_change:
                ref_aa_part, alt_aa_part = aa_change.split('/', 1)
            elif aa_change not in ['-', 'nan'] and len(aa_change) > 0:
                ref_aa_part = alt_aa_part = aa_change
            else:
                results.append((None, None))
                continue
            
            # 4. Xác minh tính hợp lệ (Ref phải khớp với FASTA)
            actual_ref_aa = full_ref_protein[pos_0based : pos_0based + len(ref_aa_part)]
            if actual_ref_aa != ref_aa_part:
                results.append((None, None))
                continue

            # 5. Tạo chuỗi Alt toàn chiều dài
            alt_full_protein = (full_ref_protein[:pos_0based] + 
                                alt_aa_part + 
                                full_ref_protein[pos_0based + len(ref_aa_part):])
            
            # Xử lý Nonsense/Stop mutation (* hoặc X)
            if '*' in alt_full_protein or 'X' in alt_aa_part:
                stop_idx = alt_full_protein.find('*')
                if stop_idx == -1: 
                    stop_idx = alt_full_protein.find('X', pos_0based)
                pad_len = max(0, len(full_ref_protein) - (stop_idx + 1))
                alt_full_protein = alt_full_protein[:stop_idx+1] + (PAD_CHAR * pad_len)

            # 6. Cắt cửa sổ 101 AA
            ref_101 = normalize_centered_protein(full_ref_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            alt_101 = normalize_centered_protein(alt_full_protein, pos_0based, TARGET_LEN, PAD_CHAR)
            
            results.append((ref_101, alt_101))
            
        except Exception:
            results.append((None, None))

    # Gán kết quả vào chunk
    chunk[f'prot_ref_seq_{TARGET_LEN}'], chunk[f'prot_alt_seq_{TARGET_LEN}'] = zip(*results)
    
    # Lọc bỏ các dòng map thất bại
    valid_chunk = chunk.dropna(subset=[f'prot_ref_seq_{TARGET_LEN}']).copy()
    
    # --- CƠ CHẾ GHI CHUNK VÀO PARQUET BẰNG PYARROW ---
    if len(valid_chunk) > 0:
        table = pa.Table.from_pandas(valid_chunk)
        
        # Ở chunk đầu tiên có dữ liệu, khởi tạo Writer
        if parquet_writer is None:
            parquet_writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
        else:
            table = table.cast(parquet_writer.schema)    
        parquet_writer.write_table(table)
        
    print(f" > Đã xử lý xong chunk {chunk_idx + 1}...")

# Đóng file an toàn
if parquet_writer:
    parquet_writer.close()

print(f"🎉 Hoàn thành! File kết quả: {OUTPUT_PATH}")

# =========================================================
# PHẦN KIỂM TRA ĐÁNH GIÁ DỮ LIỆU
# =========================================================
df = pd.read_parquet(OUTPUT_PATH)

invalid_values = [".", "-", "na", "n/a", "NA", "N/A"]
df = df.replace(invalid_values, np.nan)

print("\n=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===")
print(f"Tổng số biến thể giữ lại: {len(df):,}")

print("\n=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===")
lengths = df[f'prot_ref_seq_{TARGET_LEN}'].str.len().unique()
print(f"Các độ dài chuỗi Ref hiện có: {lengths} (Kỳ vọng: [{TARGET_LEN}])") 

check_diff = df[df[f'prot_ref_seq_{TARGET_LEN}'].str[TARGET_LEN//2] == df[f'prot_alt_seq_{TARGET_LEN}'].str[TARGET_LEN//2]]
print(f"\n=== 3. BIẾN THỂ SYNONYMOUS ===")
print(f"Số lượng biến thể có tâm {TARGET_LEN//2} giống hệt nhau (Đồng nghĩa): {len(check_diff):,}")

df

🧬 Đang nạp FASTA từ D:\vep_resources\Homo_sapiens.GRCh38.pep.all.fa...
✅ Đã nạp 382428 mã transcript.
📊 Đang xử lý file Parquet: D:\variant_data\uniprot_full_seq_after_vep_final.parquet
 > Đã xử lý xong chunk 1...
🎉 Hoàn thành! File kết quả: D:\variant_data\uniprot_full_seq_after_vep_final.parquet

=== 1. TỔNG QUAN DỮ LIỆU ĐÃ MAP THÀNH CÔNG ===
Tổng số biến thể giữ lại: 1,333

=== 2. KIỂM TRA ĐỘ DÀI CHUỖI ===
Các độ dài chuỗi Ref hiện có: [1001] (Kỳ vọng: [1001])

=== 3. BIẾN THỂ SYNONYMOUS ===
Số lượng biến thể có tâm 500 giống hệt nhau (Đồng nghĩa): 0


,Variant_ID,CHROM,POS,REF,ALT,Label,dbSNP,gene,protein_AC,aa_change,...,prot_ref_seq_31,prot_alt_seq_31,prot_ref_seq_101,prot_alt_seq_101,prot_ref_seq_201,prot_alt_seq_201,prot_ref_seq_501,prot_alt_seq_501,prot_ref_seq_1001,prot_alt_seq_1001
0,10_1072186_G_A,chr10,1072186,G,A,0,rs17856557,WDR37,Q9Y2I8,p.Ala11Thr,...,XXXXXMPTESASCSTARQTKQKRKSHSLSIR,XXXXXMPTESASCSTTRQTKQKRKSHSLSIR,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMPTESA...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMPTESA...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
1,10_1096193_A_G,chr10,1096193,A,G,0,rs2306407,WDR37,Q9Y2I8,p.Ile225Val,...,SEQLALTASGDQTAHIWRYAVQLPTPQPVAD,SEQLALTASGDQTAHVWRYAVQLPTPQPVAD,ASADHTALLWSIETGKCLVKYAGHVGSVNSIKFHPSEQLALTASGD...,ASADHTALLWSIETGKCLVKYAGHVGSVNSIKFHPSEQLALTASGD...,TYKASTSKIVSSFKTTTSRAACQLVKEYIGHRDGIWDVSVAKTQPV...,TYKASTSKIVSSFKTTTSRAACQLVKEYIGHRDGIWDVSVAKTQPV...,XXXXXXXXXXXXXXXXXXXXXXXXXXMPTESASCSTARQTKQKRKS...,XXXXXXXXXXXXXXXXXXXXXXXXXXMPTESASCSTARQTKQKRKS...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
2,10_1379131_C_A,chr10,1379131,C,A,0,rs3793733,ADARB2,Q9NS39,p.Ala44Thr,...,RSKRKDKVSILSTFLAPFKHLSPGITNTEDD,RSKRKDKVSILSTFLSPFKHLSPGITNTEDD,XXXXXXXMASVLGSGRGSGGLSSQLKCKSKRRRRRRSKRKDKVSIL...,XXXXXXXMASVLGSGRGSGGLSSQLKCKSKRRRRRRSKRKDKVSIL...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
3,10_3151324_G_A,chr10,3151324,G,A,0,rs12248937,PITRM1,Q5JRX3,p.Ala554Asp,...,EKGLELRSQQSKPQDASCLPALKVSDIEPTI,EKGLELRSQQSKPQDVSCLPALKVSDIEPTI,MRPDDKYHEKQAQVEATKLKQKVEALSPGDRQQIYEKGLELRSQQS...,MRPDDKYHEKQAQVEATKLKQKVEALSPGDRQQIYEKGLELRSQQS...,CWNHDGDPVELLKLGNQLAKFRQCLQENPKFLQEKVKQYFKNNQHK...,CWNHDGDPVELLKLGNQLAKFRQCLQENPKFLQEKVKQYFKNNQHK...,DKPREFQITCGPDSFATDPSKQTTISVSFLLPDITDTFEAFTLSLL...,DKPREFQITCGPDSFATDPSKQTTISVSFLLPDITDTFEAFTLSLL...,VTSVPELFLTAVKLTHDDTGARYLHLAREDTNNLFSVQFRTTPMDS...,VTSVPELFLTAVKLTHDDTGARYLHLAREDTNNLFSVQFRTTPMDS...
4,10_3165320_C_T,chr10,3165320,C,T,1,rs1249144069,PITRM1,Q5JRX3,p.Arg183Gln,...,FFPCLRELDFWQEGWRLEHENPSDPQTPLVF,FFPCLRELDFWQEGWQLEHENPSDPQTPLVF,TFMNAFTASDYTLYPFSTQNPKDFQNLLSVYLDATFFPCLRELDFW...,TFMNAFTASDYTLYPFSTQNPKDFQNLLSVYLDATFFPCLRELDFW...,DTNNLFSVQFRTTPMDSTGVPHILEHTVLCGSQKYPCRDPFFKMLN...,DTNNLFSVQFRTTPMDSTGVPHILEHTVLCGSQKYPCRDPFFKMLN...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1328,20_63495972_C_T,chr20,63495972,C,T,1,rs587777162,EEF1A2,Q05639,p.Gly70Ser,...,KYAWVLDKLKAERERGITIDISLWKFETTKY,KYAWVLDKLKAERERSITIDISLWKFETTKY,KSTTTGHLIYKCGGIDKRTIEKFEKEAAEMGKGSFKYAWVLDKLKA...,KSTTTGHLIYKCGGIDKRTIEKFEKEAAEMGKGSFKYAWVLDKLKA...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMGKEKTHINIVVIGH...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXMGKEKTHINIVVIGH...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...
1329,20_63547241_C_A,chr20,63547241,C,A,0,rs55863722,SRMS,Q9H3Y6,p.Gly75Arg,...,YDFTARCGGELSVRRGDRLCALEEGGGYIFA,YDFTARCGGELSVRRWDRLCALEEGGGYIFA,EPDHGTPGSLDPNTDPVPTLPAEPCSPFPQLFLALYDFTARCGGEL...,EPDHGTPGSLDPNTDPVPTLPAEPCSPFPQLFLALYDFTARCGGEL...,XXXXXXXXXXXXXXXXXXXXXXXXXXMEPFLRRRLAFLSFFWDKIW...,XXXXXXXXXXXXXXXXXXXXXXXXXXMEPFLRRRLAFLSFFWDKIW...,XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX...,XXXXXXXXXXXXXXXXXXXXXXX